# Model Comparison: 1D CNN vs. Multi-resolution CNN

## 1. Load Evaluation Results

This notebook compares two CNN architectures for
MS/MS-based molecular fingerprint prediction:

1. Standard 1D CNN
2. Multi-resolution CNN

Both models were trained using:
- The same 17,456 training spectra
- The same 2,000 validation spectra
- 1024-bit Morgan fingerprint targets
- Weighted BCE loss
- AdamW optimizer
- Learning rate of 0.0003

We compare:

- Validation loss
- Fingerprint Micro F1
- Mean Tanimoto similarity
- Candidate Ranking MRR@25
- Mass-only and random ranking baselines

The ranking experiments use an oracle candidate library
and should not be interpreted as final Kaggle scores.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current

# Set these environment variables to override the local defaults.
REPO_ROOT = Path(
    os.environ.get("CASMI_REPO_ROOT", find_repo_root())
).expanduser().resolve()
SAVE_DIR = Path(
    os.environ.get(
        "CASMI_MEMBER2_DIR",
        REPO_ROOT / "outputs" / "member2"
    )
).expanduser().resolve()
SAVE_DIR.mkdir(parents=True, exist_ok=True)

EVAL_DIR = SAVE_DIR / "evaluation"
DATA_DIR = SAVE_DIR / "prototype_data"

print("Evaluation directory:", EVAL_DIR)
print("Environment ready!")

In [ ]:
# Load evaluation summaries
with open(EVAL_DIR / "cnn1d_evaluation_summary.json") as f:
    cnn1d_summary = json.load(f)

with open(EVAL_DIR / "multires_cnn_evaluation_summary.json") as f:
    multires_summary = json.load(f)


# Load threshold results
cnn1d_thresholds = pd.read_csv(
    EVAL_DIR / "cnn1d_threshold_results.csv"
)

multires_thresholds = pd.read_csv(
    EVAL_DIR / "multires_cnn_threshold_results.csv"
)


# Load continuous fingerprint probabilities
with np.load(
    EVAL_DIR / "cnn1d_val_predictions.npz",
    allow_pickle=False
) as data:
    cnn1d_probs = data["probabilities"]
    cnn1d_targets = data["targets"]
    cnn1d_keys = data["inchikey14"]
    cnn1d_smiles = data["smiles"]

with np.load(
    EVAL_DIR / "multires_cnn_val_predictions.npz",
    allow_pickle=False
) as data:
    multires_probs = data["probabilities"]
    multires_targets = data["targets"]
    multires_keys = data["inchikey14"]
    multires_smiles = data["smiles"]


# Load validation data
df_val = pd.read_parquet(
    DATA_DIR / "val_prototype.parquet"
)


# Verify exact alignment
assert cnn1d_probs.shape == multires_probs.shape
assert cnn1d_probs.shape == (2000, 1024)

assert np.array_equal(cnn1d_keys, multires_keys)
assert np.array_equal(cnn1d_smiles, multires_smiles)
assert np.array_equal(cnn1d_targets, multires_targets)

assert np.array_equal(
    cnn1d_keys,
    df_val["inchikey14"].to_numpy(dtype=str)
)

assert np.array_equal(
    cnn1d_smiles,
    df_val["normalized_smiles"].to_numpy(dtype=str)
)

print("1D CNN predictions:", cnn1d_probs.shape)
print("Multi-resolution predictions:", multires_probs.shape)
print("Validation spectra:", len(df_val))

print("\nBoth models loaded and aligned successfully!")

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "1D CNN",
        "Parameters": 1610112,
        "Best Epoch": cnn1d_summary["checkpoint_epoch"],
        "Val Loss": cnn1d_summary["best_val_loss"],
        "Best F1": cnn1d_summary["best_f1"],
        "F1 Threshold": cnn1d_summary["best_f1_threshold"],
        "Best Tanimoto": cnn1d_summary["best_tanimoto"]
    },
    {
        "Model": "Multi-resolution CNN",
        "Parameters": 1615456,
        "Best Epoch": multires_summary["checkpoint_epoch"],
        "Val Loss": multires_summary["best_val_loss"],
        "Best F1": multires_summary["best_f1"],
        "F1 Threshold": multires_summary["best_f1_threshold"],
        "Best Tanimoto": multires_summary["best_tanimoto"]
    }
])

display(comparison.round(4))

print("Fingerprint comparison ready!")

## 2. Shared Candidate Library

We construct a shared oracle candidate library from the
validation dataset for both CNN architectures.

### Candidate Representation
- One candidate per original structure key
- 1024-bit Morgan fingerprint (radius=2)
- Exact neutral molecular mass calculated using RDKit

### Query Representation
- Mean aggregation of predicted fingerprint probabilities
  across spectra belonging to the same structure
- Neutral mass estimated from precursor m/z and adduct

Structures with multiple SMILES representations are excluded
from this initial oracle ranking experiment.

Both models use the exact same candidate library,
query structures, and mass-filtering conditions.

This is a diagnostic oracle benchmark, not a final
Kaggle competition score.

In [ ]:
# Install RDKit if needed
!pip -q install "rdkit==2026.3.3"

In [ ]:
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator

assert rdBase.rdkitVersion == "2026.03.3"

FP_SIZE = 1024
FP_RADIUS = 2

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_SIZE,
    includeChirality=False
)

# 1. Identify groups with a single SMILES representation
smiles_counts = (
    df_val.groupby("inchikey14")["normalized_smiles"]
    .nunique(dropna=False)
)

consistent_keys = set(
    smiles_counts[smiles_counts == 1].index
)

candidate_df = (
    df_val[
        df_val["inchikey14"].isin(consistent_keys)
    ][["inchikey14", "normalized_smiles"]]
    .drop_duplicates(subset=["inchikey14"])
    .sort_values("inchikey14")
    .reset_index(drop=True)
)

# 2. Generate candidate fingerprints and neutral masses
candidate_keys = []
candidate_fps = []
candidate_masses = []

for row in candidate_df.itertuples(index=False):

    mol = Chem.MolFromSmiles(row.normalized_smiles)

    if mol is None:
        continue

    fp = morgan_generator.GetFingerprint(mol)

    fp_array = np.zeros(FP_SIZE, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, fp_array)

    candidate_keys.append(row.inchikey14)
    candidate_fps.append(fp_array)
    candidate_masses.append(Descriptors.ExactMolWt(mol))

candidate_keys = np.asarray(candidate_keys)

candidate_fps = np.stack(
    candidate_fps
).astype(np.float32)

candidate_masses = np.asarray(
    candidate_masses,
    dtype=np.float64
)

# 3. Aggregate spectrum-level predictions by structure
group_indices = (
    df_val.groupby("inchikey14", sort=True).indices
)

def aggregate_predictions(probabilities):
    return np.stack([
        probabilities[group_indices[key]].mean(axis=0)
        for key in candidate_keys
    ]).astype(np.float32)


cnn1d_query_probs = aggregate_predictions(cnn1d_probs)

multires_query_probs = aggregate_predictions(multires_probs)

# Verify identical alignment
assert cnn1d_query_probs.shape == multires_query_probs.shape
assert cnn1d_query_probs.shape == candidate_fps.shape
assert len(candidate_keys) == len(set(candidate_keys))

print("Candidate library:", candidate_fps.shape)
print("1D CNN queries:", cnn1d_query_probs.shape)
print("Multi-resolution queries:", multires_query_probs.shape)
print("Candidate neutral masses:", candidate_masses.shape)

print("\nShared candidate library ready!")

## 3. Mass-Filtered Candidate Ranking Comparison

We compare the two CNN architectures using identical
oracle candidates and precursor-mass constraints.

### Ranking Methods
1. Mass-only ranking
2. Standard 1D CNN with Soft Tanimoto similarity
3. Multi-resolution CNN with Soft Tanimoto similarity
4. Expected random ranking within the mass filter

### Evaluation
- Mass tolerance: 20 ppm and 50 ppm
- MRR@25
- Hit@1, Hit@5, Hit@25
- Candidate retention rate

We also evaluate queries with at least two mass-matched
candidates to better measure structural discrimination.

Queries whose correct candidate is excluded by mass
filtering receive zero reciprocal-rank contribution.

In [ ]:
# Supported adducts
PROTON_MASS = 1.007276466621

ADDUCT_OFFSETS = {
    "[M+H]+": PROTON_MASS,
    "[M-H]-": -PROTON_MASS
}

# Estimate neutral molecular mass from observed spectra
observations = df_val[
    ["inchikey14", "precursor_mz", "adduct"]
].copy()

observations["precursor_mz"] = pd.to_numeric(
    observations["precursor_mz"],
    errors="coerce"
)

observations["offset"] = (
    observations["adduct"].map(ADDUCT_OFFSETS)
)

observations["neutral_mass_est"] = (
    observations["precursor_mz"] -
    observations["offset"]
)

valid_observations = observations[
    observations["neutral_mass_est"].notna()
    & np.isfinite(observations["neutral_mass_est"])
    & (observations["neutral_mass_est"] > 0)
]

mass_by_key = (
    valid_observations
    .groupby("inchikey14")["neutral_mass_est"]
    .median()
    .to_dict()
)

query_masses = np.asarray(
    [mass_by_key.get(key, np.nan) for key in candidate_keys],
    dtype=np.float64
)

eligible_idx = np.flatnonzero(
    np.isfinite(query_masses)
)

# ppm difference between each query and candidate
ppm_errors = (
    np.abs(
        query_masses[eligible_idx, None]
        - candidate_masses[None, :]
    )
    / candidate_masses[None, :]
    * 1e6
)

print("Total oracle queries:", len(candidate_keys))
print("Mass-eligible queries:", len(eligible_idx))
print("Queries without supported mass:",
      len(candidate_keys) - len(eligible_idx))

print("Mass error matrix:", ppm_errors.shape)

print("\nMass filtering preparation completed!")

In [ ]:
# Soft Tanimoto using continuous fingerprint probabilities
def soft_tanimoto_scores(query_probs, candidate_fps):

    intersection = query_probs @ candidate_fps.T

    query_sums = query_probs.sum(axis=1, keepdims=True)
    candidate_sums = candidate_fps.sum(axis=1)[None, :]

    denominator = query_sums + candidate_sums - intersection

    return intersection / np.maximum(denominator, 1e-8)


# Scores for both CNN models
cnn1d_scores = soft_tanimoto_scores(
    cnn1d_query_probs, candidate_fps
)

multires_scores = soft_tanimoto_scores(
    multires_query_probs, candidate_fps
)


def get_correct_ranks(score_matrix, accepted, true_indices):

    filtered_scores = np.where(
        accepted, score_matrix, -np.inf
    )

    ranked = np.argsort(
        -filtered_scores,
        axis=1,
        kind="stable"
    )

    positions = np.argsort(
        ranked,
        axis=1,
        kind="stable"
    )

    rows = np.arange(len(true_indices))

    ranks = (
        positions[rows, true_indices].astype(float) + 1
    )

    retained = accepted[rows, true_indices]

    ranks[~retained] = np.inf

    return ranks


def ranking_metrics(ranks):

    ranks = np.asarray(ranks)

    return {
        "MRR@25": np.mean(
            np.where(ranks <= 25, 1.0 / ranks, 0.0)
        ),
        "Hit@1": np.mean(ranks <= 1),
        "Hit@5": np.mean(ranks <= 5),
        "Hit@25": np.mean(ranks <= 25)
    }


ranking_results = []

for tolerance in [20, 50]:

    accepted = ppm_errors <= tolerance
    candidate_counts = accepted.sum(axis=1)

    rows = np.arange(len(eligible_idx))
    true_retained = accepted[rows, eligible_idx]

    # Mass-only scores: smaller mass error is better
    mass_ranks = get_correct_ranks(
        -ppm_errors,
        accepted,
        eligible_idx
    )

    # Model 1
    cnn1d_ranks = get_correct_ranks(
        cnn1d_scores[eligible_idx],
        accepted,
        eligible_idx
    )

    # Model 2
    multires_ranks = get_correct_ranks(
        multires_scores[eligible_idx],
        accepted,
        eligible_idx
    )

    # Exact expected random MRR within each candidate set
    random_rr = np.zeros(len(eligible_idx), dtype=float)

    for i, n in enumerate(candidate_counts):
        if true_retained[i] and n > 0:
            random_rr[i] = (
                sum(1.0 / r for r in range(1, min(n, 25) + 1))
                / n
            )

    methods = {
        "Mass-only": mass_ranks,
        "1D CNN": cnn1d_ranks,
        "Multi-resolution CNN": multires_ranks
    }

    for subset_name, mask in [
        ("all_eligible", np.ones(len(eligible_idx), dtype=bool)),
        ("multiple_candidates", candidate_counts >= 2)
    ]:

        for method_name, ranks in methods.items():

            metrics = ranking_metrics(ranks[mask])

            ranking_results.append({
                "Model": method_name,
                "Tolerance (ppm)": tolerance,
                "Subset": subset_name,
                "Queries": int(mask.sum()),
                "MRR@25": metrics["MRR@25"],
                "Hit@1": metrics["Hit@1"],
                "Hit@5": metrics["Hit@5"],
                "Hit@25": metrics["Hit@25"],
                "True Retention": true_retained[mask].mean()
            })

        # Add expected random MRR reference
        ranking_results.append({
            "Model": "Random expected",
            "Tolerance (ppm)": tolerance,
            "Subset": subset_name,
            "Queries": int(mask.sum()),
            "MRR@25": random_rr[mask].mean(),
            "Hit@1": np.nan,
            "Hit@5": np.nan,
            "Hit@25": np.nan,
            "True Retention": true_retained[mask].mean()
        })

ranking_comparison_df = pd.DataFrame(ranking_results)

display(
    ranking_comparison_df.round(4)
)

print("\nModel ranking comparison completed!")

## 4. Paired Ranking Comparison

The two CNN models achieved nearly identical
candidate-ranking performance.

To examine whether the small difference is consistent,
we compare their reciprocal ranks on exactly the same
validation queries.

We use paired bootstrap resampling to estimate a
95% confidence interval for the difference in MRR@25:

Multi-resolution CNN minus Standard 1D CNN.

We evaluate both:
- All mass-eligible queries
- Queries with at least two mass-matched candidates

The confidence interval describes uncertainty within
this oracle validation benchmark. It does not measure
generalization to the actual competition test set.

In [ ]:
# Paired bootstrap comparison of ranking performance

def reciprocal_rank_at_25(ranks):
    ranks = np.asarray(ranks, dtype=float)

    rr = np.zeros(len(ranks), dtype=float)
    valid = ranks <= 25

    rr[valid] = 1.0 / ranks[valid]

    return rr


rng = np.random.default_rng(42)
comparison_records = []

for tolerance in [20, 50]:

    accepted = ppm_errors <= tolerance
    candidate_counts = accepted.sum(axis=1)

    # Same queries and candidates for both models
    cnn1d_ranks = get_correct_ranks(
        cnn1d_scores[eligible_idx],
        accepted,
        eligible_idx
    )

    multires_ranks = get_correct_ranks(
        multires_scores[eligible_idx],
        accepted,
        eligible_idx
    )

    rr1 = reciprocal_rank_at_25(cnn1d_ranks)
    rr2 = reciprocal_rank_at_25(multires_ranks)

    for subset_name, mask in [
        ("all_eligible", np.ones(len(eligible_idx), dtype=bool)),
        ("multiple_candidates", candidate_counts >= 2)
    ]:

        # Difference: Model 2 - Model 1
        differences = rr2[mask] - rr1[mask]

        observed_difference = differences.mean()

        # Paired bootstrap: sample queries with replacement
        n = len(differences)

        boot_indices = rng.integers(
            0, n, size=(2000, n)
        )

        boot_means = differences[boot_indices].mean(axis=1)

        ci_low, ci_high = np.percentile(
            boot_means, [2.5, 97.5]
        )

        comparison_records.append({
            "tolerance_ppm": tolerance,
            "subset": subset_name,
            "queries": n,
            "cnn1d_mrr25": rr1[mask].mean(),
            "multires_mrr25": rr2[mask].mean(),
            "mrr_difference": observed_difference,
            "ci_95_low": ci_low,
            "ci_95_high": ci_high,
            "multires_better_queries": int(
                np.sum(differences > 0)
            ),
            "cnn1d_better_queries": int(
                np.sum(differences < 0)
            ),
            "tied_queries": int(
                np.sum(differences == 0)
            )
        })


paired_comparison_df = pd.DataFrame(comparison_records)

display(paired_comparison_df.round(5))

# Save ranking and paired comparison results
ranking_comparison_df.to_csv(
    EVAL_DIR / "final_ranking_comparison.csv",
    index=False
)

paired_comparison_df.to_csv(
    EVAL_DIR / "paired_ranking_comparison.csv",
    index=False
)

print("\nPaired comparison completed and saved!")

## 5. Results and Discussion

### 5.1 Fingerprint Prediction

We compared a standard 1D CNN and a Multi-resolution CNN
using the same training and validation datasets.

The Multi-resolution CNN achieved a slightly lower
validation loss (0.8478 vs. 0.8492). However, the standard
1D CNN achieved a higher best Micro F1 (0.3163 vs. 0.3099)
and mean Tanimoto similarity (0.1863 vs. 0.1834) after
threshold tuning.

Overall, the two architectures showed similar
fingerprint prediction performance.

### 5.2 Candidate Ranking

We evaluated both models using an oracle candidate
library with precursor mass filtering.

At 20 ppm, the standard 1D CNN achieved an MRR@25 of
0.8540, while the Multi-resolution CNN achieved 0.8532.

At 50 ppm, their MRR@25 scores were 0.7521 and 0.7512,
respectively.

The Mass-only baseline achieved higher MRR@25 scores
than either CNN method.

This suggests that precursor mass was more informative
than the predicted fingerprints in our current
small candidate library.

### 5.3 Paired Bootstrap Analysis

We used paired bootstrap resampling to compare the
MRR@25 scores of the two models.

The 95% confidence intervals for the MRR differences
included zero at both 20 ppm and 50 ppm.

Most validation queries also had identical correct
candidate ranks under the two models.

Therefore, we did not find clear evidence that the
Multi-resolution CNN improved candidate ranking
performance over the standard 1D CNN.

### 5.4 Limitations

This study has several limitations.

First, we used only about 17,000 training spectra
instead of the complete competition dataset.

Second, our candidate library was relatively small
and included the correct validation structures.
Many queries had only one or two candidates after
mass filtering.

Third, the validation dataset was based on a random
spectrum sample and contained fewer spectra per
molecule than the competition test dataset.

Finally, we evaluated only one training seed for
each architecture.

Therefore, the oracle ranking results should not
be interpreted as actual Kaggle competition scores.

### 5.5 Conclusion

Both CNN architectures learned some useful molecular
fingerprint information from MS/MS spectra.

However, the Multi-resolution CNN did not provide a
clear improvement over the simpler 1D CNN.

For the current prototype, the standard 1D CNN is
a reasonable baseline because it has similar
performance and lower training cost.

Future work should focus on larger training datasets,
more challenging candidate libraries, improved
fingerprint prediction, and realistic molecule-level
ranking evaluation.